In [1]:
import pandas as pd
import faiss
import joblib
from sentence_transformers import SentenceTransformer

CSV_PATH = './Data/dataset.csv'
EMBEDDING_MODEL = 'keepitreal/vietnamese-sbert'

simu = 0.8

K:\Deeplearning\ChatbotAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

from Lmd_Utils import cleantext
df = pd.read_parquet('./Data/train.parquet')
df2 = pd.read_parquet('./Data/validation.parquet')
df = pd.concat([df, df2], ignore_index=True)
questions, answers = [], []
for i in range(len(df)):
    is_imp = df['is_impossible'].iloc[i]
    ans_dict = df['answers'].iloc[i]
    plausible_dict = df['plausible_answers'].iloc[i]
    ques_text = df['question'].iloc[i]
    if is_imp == True:
        if isinstance(plausible_dict, dict) and len(plausible_dict.get('text', [])) > 0:
            answers.append(cleantext(plausible_dict['text'][0]))
            questions.append(cleantext(ques_text))
    else:
        if isinstance(ans_dict, dict) and len(ans_dict.get('text', [])) > 0:
            answers.append(cleantext(ans_dict['text'][0]))
            questions.append(cleantext(ques_text))
df = pd.DataFrame({'question': questions, 'answer': answers})


df = df[['question', 'answer']].dropna()
df.columns = ['question', 'answer']

questions = df['question'].tolist()
answers = df['answer'].tolist()



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lemin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lemin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35213.54it/s]
RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:

embed_model = SentenceTransformer(EMBEDDING_MODEL)

question_embeddings = embed_model.encode(
    questions,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f'✅ Embedding shape: {question_embeddings.shape}')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 36182.87it/s]
RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 505/505 [05:40<00:00,  1.48it/s]

✅ Embedding shape: (32268, 768)


In [4]:

faiss.normalize_L2(question_embeddings)
n = question_embeddings.shape[1]
index = faiss.IndexFlatIP(n)
index.add(question_embeddings)


faiss.write_index(index, './models_rag/faiss_index.bin')
joblib.dump(answers, './models_rag/answers.pkl')
joblib.dump(questions, './models_rag/questions.pkl')


['./models_rag/questions.pkl']